# Optimización Final con Optuna

**Proyecto:** Sistema de Clasificación de Acciones S&P 500

**Grupo 27** - Universidad de Los Andes

---

## Objetivo

Optimizar hiperparámetros de los 3 mejores modelos del notebook 02:
1. **Logistic Regression + Top-5 features** (ROC-AUC baseline: 0.511)
2. **XGBoost + PCA-15** (ROC-AUC baseline: 0.507)
3. **Random Forest + PCA-5** (ROC-AUC baseline: 0.504)

**Métrica de optimización:** ROC-AUC (más robusta que F1-Score)

In [2]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import optuna

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

import joblib
import warnings
warnings.filterwarnings('ignore')

/Users/santiagobeltransalazar/Desktop/Master en Inteligencia Analítica de Datos/Tercer semestre/Despliegue de soluciones/Despliegue-Soluciones-Proyecto/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configuración

In [3]:
# Configurar MLflow
experiment_name = "/sp500-optuna-optimization"
mlflow.set_experiment(experiment_name)

print(f"Experimento: {experiment_name}")

2025/11/07 22:10:41 INFO mlflow.tracking.fluent: Experiment with name '/sp500-optuna-optimization' does not exist. Creating a new experiment.


Experimento: /sp500-optuna-optimization


## 2. Carga de Datos

In [4]:
# Cargar datasets
train = pd.read_parquet('../../data/processed/ml_ready/train.parquet')
test = pd.read_parquet('../../data/processed/ml_ready/test.parquet')

# Separar features y target
feature_cols = [col for col in train.columns if col not in ['Ticker', 'Date', 'Target']]

X_train_full = train[feature_cols]
y_train = train['Target']
X_test_full = test[feature_cols]
y_test = test['Target']

print(f"Datos cargados: {X_train_full.shape[0]} train, {X_test_full.shape[0]} test")
print(f"Features: {len(feature_cols)}")

Datos cargados: 20128 train, 5032 test
Features: 17


## 3. Preparar las 3 Configuraciones

Basadas en los mejores resultados del notebook 02.

In [5]:
# Top-5 features (para Logistic Regression)
from sklearn.ensemble import RandomForestClassifier as RFC
rf_temp = RFC(random_state=42)
rf_temp.fit(X_train_full, y_train)

feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_temp.feature_importances_
}).sort_values('importance', ascending=False)

top_5_features = feature_importance.head(5)['feature'].tolist()

print("Top-5 features:")
print(top_5_features)

Top-5 features:
['Volume_change', 'Returns', 'RSI_14', 'Volatility_10', 'MACD_diff']


In [6]:
# Configuración 1: Logistic Regression + Top-5
X_train_lr = X_train_full[top_5_features]
X_test_lr = X_test_full[top_5_features]

print(f"Config 1 - Logistic Regression: {X_train_lr.shape}")

# Configuración 2: XGBoost + PCA-15
scaler_xgb = StandardScaler()
X_train_scaled_xgb = scaler_xgb.fit_transform(X_train_full)
X_test_scaled_xgb = scaler_xgb.transform(X_test_full)

pca_xgb = PCA(n_components=15, random_state=42)
X_train_xgb = pca_xgb.fit_transform(X_train_scaled_xgb)
X_test_xgb = pca_xgb.transform(X_test_scaled_xgb)

print(f"Config 2 - XGBoost: {X_train_xgb.shape}, Varianza: {pca_xgb.explained_variance_ratio_.sum():.2%}")

# Configuración 3: Random Forest + PCA-5
scaler_rf = StandardScaler()
X_train_scaled_rf = scaler_rf.fit_transform(X_train_full)
X_test_scaled_rf = scaler_rf.transform(X_test_full)

pca_rf = PCA(n_components=5, random_state=42)
X_train_rf = pca_rf.fit_transform(X_train_scaled_rf)
X_test_rf = pca_rf.transform(X_test_scaled_rf)

print(f"Config 3 - Random Forest: {X_train_rf.shape}, Varianza: {pca_rf.explained_variance_ratio_.sum():.2%}")

Config 1 - Logistic Regression: (20128, 5)
Config 2 - XGBoost: (20128, 15), Varianza: 100.00%
Config 3 - Random Forest: (20128, 5), Varianza: 84.91%


## 4. Optimización 1: Logistic Regression + Top-5

In [ ]:
def objective_lr(trial):
    """Optuna objective para Logistic Regression."""
    params = {
        'C': trial.suggest_float('C', 0.001, 100, log=True),
        'penalty': trial.suggest_categorical('penalty', ['l1', 'l2']),
        'solver': 'saga',
        'max_iter': 1000,
        'random_state': 42
    }
    
    model = LogisticRegression(**params)
    
    # Cross-validation con ROC-AUC
    cv_scores = cross_val_score(
        model, X_train_lr, y_train,
        cv=3, scoring='roc_auc', n_jobs=-1
    )
    
    return cv_scores.mean()

print("Optimizando Logistic Regression + Top-5...")
study_lr = optuna.create_study(direction='maximize')
study_lr.optimize(objective_lr, n_trials=50, show_progress_bar=True)

print(f"\nMejor ROC-AUC (CV): {study_lr.best_value:.4f}")
print(f"Mejores parámetros: {study_lr.best_params}")

[I 2025-11-07 22:10:51,532] A new study created in memory with name: no-name-f1c2f576-566b-4a08-8a95-006db3354ecd


Optimizando Logistic Regression + Top-5...


Best trial: 0. Best value: 0.506742:   2%|▏         | 1/50 [00:02<01:55,  2.36s/it]

[I 2025-11-07 22:10:53,902] Trial 0 finished with value: 0.5067420334666698 and parameters: {'C': 7.159189587167485, 'penalty': 'l2'}. Best is trial 0 with value: 0.5067420334666698.


Best trial: 0. Best value: 0.506742:   4%|▍         | 2/50 [00:04<01:40,  2.09s/it]

...

[I 2025-11-07 22:11:17,202] Trial 45 finished with value: 0.5053272835897366 and parameters: {'C': 0.021536899131236437, 'penalty': 'l1'}. Best is trial 22 with value: 0.5094243893970005.
[I 2025-11-07 22:11:17,350] Trial 46 finished with value: 0.5035124116846436 and parameters: {'C': 0.005203399294830923, 'penalty': 'l2'}. Best is trial 22 with value: 0.5094243893970005.


Best trial: 22. Best value: 0.509424:  96%|█████████▌| 48/50 [00:26<00:00,  2.92it/s]

[I 2025-11-07 22:11:18,103] Trial 47 finished with value: 0.5046426637512608 and parameters: {'C': 0.4588093479061346, 'penalty': 'l1'}. Best is trial 22 with value: 0.5094243893970005.


Best trial: 22. Best value: 0.509424: 100%|██████████| 50/50 [00:26<00:00,  1.86it/s]

[I 2025-11-07 22:11:18,314] Trial 48 finished with value: 0.5090053746838645 and parameters: {'C': 0.010927534611475736, 'penalty': 'l1'}. Best is trial 22 with value: 0.5094243893970005.
[I 2025-11-07 22:11:18,466] Trial 49 finished with value: 0.5036845908996476 and parameters: {'C': 0.0026996557568181065, 'penalty': 'l1'}. Best is trial 22 with value: 0.5094243893970005.

Mejor ROC-AUC (CV): 0.5094
Mejores parámetros: {'C': 0.005027020107643563, 'penalty': 'l1'}


## 5. Optimización 2: XGBoost + PCA-15

In [8]:
def objective_xgb(trial):
    """Optuna objective para XGBoost."""
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'random_state': 42,
        'eval_metric': 'logloss'
    }
    
    model = XGBClassifier(**params)
    
    cv_scores = cross_val_score(
        model, X_train_xgb, y_train,
        cv=3, scoring='roc_auc', n_jobs=-1
    )
    
    return cv_scores.mean()

print("Optimizando XGBoost + PCA-15...")
study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=100, show_progress_bar=True)

print(f"\nMejor ROC-AUC (CV): {study_xgb.best_value:.4f}")
print(f"Mejores parámetros: {study_xgb.best_params}")

[I 2025-11-07 22:11:18,481] A new study created in memory with name: no-name-147d1204-9e57-4c3a-9f9f-f5a68ef46a55


Optimizando XGBoost + PCA-15...


Best trial: 0. Best value: 0.512756:   1%|          | 1/100 [00:00<00:22,  4.48it/s]

[I 2025-11-07 22:11:18,705] Trial 0 finished with value: 0.5127557238371446 and parameters: {'n_estimators': 91, 'max_depth': 4, 'learning_rate': 0.11564826830171483, 'subsample': 0.8709184969808533, 'colsample_bytree': 0.6578892244889617, 'min_child_weight': 8, 'gamma': 4.057631839144869}. Best is trial 0 with value: 0.5127557238371446.


Best trial: 1. Best value: 0.513182:   2%|▏         | 2/100 [00:00<00:28,  3.45it/s]

...

[I 2025-11-07 22:11:47,337] Trial 97 finished with value: 0.5143766760513903 and parameters: {'n_estimators': 257, 'max_depth': 4, 'learning_rate': 0.07313661709510029, 'subsample': 0.6117556756633039, 'colsample_bytree': 0.6429262481102401, 'min_child_weight': 10, 'gamma': 4.776597987412873}. Best is trial 8 with value: 0.5226609162753323.


Best trial: 8. Best value: 0.522661:  99%|█████████▉| 99/100 [00:29<00:00,  3.69it/s]

[I 2025-11-07 22:11:47,549] Trial 98 finished with value: 0.5126471422517972 and parameters: {'n_estimators': 292, 'max_depth': 7, 'learning_rate': 0.2182676805562643, 'subsample': 0.6645652198270926, 'colsample_bytree': 0.9018290136231031, 'min_child_weight': 8, 'gamma': 4.2851532595848525}. Best is trial 8 with value: 0.5226609162753323.


Best trial: 8. Best value: 0.522661: 100%|██████████| 100/100 [00:29<00:00,  3.40it/s]

[I 2025-11-07 22:11:47,895] Trial 99 finished with value: 0.5151167277941624 and parameters: {'n_estimators': 244, 'max_depth': 5, 'learning_rate': 0.09599144219563983, 'subsample': 0.6330565050539289, 'colsample_bytree': 0.819670931577401, 'min_child_weight': 5, 'gamma': 3.3139739226529263}. Best is trial 8 with value: 0.5226609162753323.

Mejor ROC-AUC (CV): 0.5227
Mejores parámetros: {'n_estimators': 166, 'max_depth': 5, 'learning_rate': 0.08156132554133447, 'subsample': 0.6536122368071011, 'colsample_bytree': 0.9138889181283119, 'min_child_weight': 4, 'gamma': 4.477319114506578}


## 6. Optimización 3: Random Forest + PCA-5

In [9]:
def objective_rf(trial):
    """Optuna objective para Random Forest."""
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        'random_state': 42,
        'n_jobs': -1
    }
    
    model = RandomForestClassifier(**params)
    
    cv_scores = cross_val_score(
        model, X_train_rf, y_train,
        cv=3, scoring='roc_auc', n_jobs=-1
    )
    
    return cv_scores.mean()

print("Optimizando Random Forest + PCA-5...")
study_rf = optuna.create_study(direction='maximize')
study_rf.optimize(objective_rf, n_trials=50, show_progress_bar=True)

print(f"\nMejor ROC-AUC (CV): {study_rf.best_value:.4f}")
print(f"Mejores parámetros: {study_rf.best_params}")

[I 2025-11-07 22:11:47,910] A new study created in memory with name: no-name-9ef2bacb-2926-49c4-9c5a-d9071a24b5a2


Optimizando Random Forest + PCA-5...


Best trial: 0. Best value: 0.504032:   2%|▏         | 1/50 [00:02<01:58,  2.42s/it]

[I 2025-11-07 22:11:50,334] Trial 0 finished with value: 0.5040324569266391 and parameters: {'n_estimators': 282, 'max_depth': 13, 'min_samples_split': 8, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.5040324569266391.


Best trial: 1. Best value: 0.504775:   4%|▍         | 2/50 [00:03<01:28,  1.84s/it]

...

[I 2025-11-07 22:12:36,566] Trial 47 finished with value: 0.5040856079792471 and parameters: {'n_estimators': 199, 'max_depth': 12, 'min_samples_split': 11, 'min_samples_leaf': 2, 'max_features': 'log2'}. Best is trial 36 with value: 0.5127111817385995.


Best trial: 36. Best value: 0.512711:  98%|█████████▊| 49/50 [00:50<00:01,  1.12s/it]

[I 2025-11-07 22:12:38,281] Trial 48 finished with value: 0.4997707193132181 and parameters: {'n_estimators': 166, 'max_depth': 20, 'min_samples_split': 18, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 36 with value: 0.5127111817385995.


Best trial: 36. Best value: 0.512711: 100%|██████████| 50/50 [00:51<00:00,  1.03s/it]

[I 2025-11-07 22:12:39,210] Trial 49 finished with value: 0.5068589328932848 and parameters: {'n_estimators': 179, 'max_depth': 7, 'min_samples_split': 19, 'min_samples_leaf': 6, 'max_features': 'log2'}. Best is trial 36 with value: 0.5127111817385995.

Mejor ROC-AUC (CV): 0.5127
Mejores parámetros: {'n_estimators': 172, 'max_depth': 3, 'min_samples_split': 16, 'min_samples_leaf': 1, 'max_features': 'log2'}


## 7. Comparación de los 3 Modelos Optimizados

Entrenar cada modelo con sus mejores hiperparámetros y evaluar en test set.

In [15]:
# Entrenar y evaluar los 3 modelos optimizados
resultados_optuna = []

# Modelo 1: Logistic Regression
params_lr = study_lr.best_params.copy()
params_lr.update({'solver': 'saga', 'max_iter': 1000, 'random_state': 42})
model_lr = LogisticRegression(**params_lr)
model_lr.fit(X_train_lr, y_train)

y_pred_lr = model_lr.predict(X_test_lr)
y_pred_proba_lr = model_lr.predict_proba(X_test_lr)[:, 1]

resultados_optuna.append({
    'Modelo': 'Logistic Regression',
    'Config': 'Top-5',
    'ROC-AUC (CV)': study_lr.best_value,
    'ROC-AUC (Test)': roc_auc_score(y_test, y_pred_proba_lr),
    'Accuracy': accuracy_score(y_test, y_pred_lr),
    'F1-Score': f1_score(y_test, y_pred_lr)
})

# Modelo 2: XGBoost
params_xgb = study_xgb.best_params.copy()
params_xgb.update({'random_state': 42, 'eval_metric': 'logloss'})
model_xgb = XGBClassifier(**params_xgb)
model_xgb.fit(X_train_xgb, y_train)

y_pred_xgb = model_xgb.predict(X_test_xgb)
y_pred_proba_xgb = model_xgb.predict_proba(X_test_xgb)[:, 1]

resultados_optuna.append({
    'Modelo': 'XGBoost',
    'Config': 'PCA-15',
    'ROC-AUC (CV)': study_xgb.best_value,
    'ROC-AUC (Test)': roc_auc_score(y_test, y_pred_proba_xgb),
    'Accuracy': accuracy_score(y_test, y_pred_xgb),
    'F1-Score': f1_score(y_test, y_pred_xgb)
})

# Modelo 3: Random Forest
params_rf = study_rf.best_params.copy()
params_rf.update({'random_state': 42, 'n_jobs': -1})
model_rf = RandomForestClassifier(**params_rf)
model_rf.fit(X_train_rf, y_train)

y_pred_rf = model_rf.predict(X_test_rf)
y_pred_proba_rf = model_rf.predict_proba(X_test_rf)[:, 1]

resultados_optuna.append({
    'Modelo': 'Random Forest',
    'Config': 'PCA-5',
    'ROC-AUC (CV)': study_rf.best_value,
    'ROC-AUC (Test)': roc_auc_score(y_test, y_pred_proba_rf),
    'Accuracy': accuracy_score(y_test, y_pred_rf),
    'F1-Score': f1_score(y_test, y_pred_rf)
})

# Mostrar comparación
df_resultados = pd.DataFrame(resultados_optuna)
df_resultados = df_resultados.sort_values('ROC-AUC (Test)', ascending=False).reset_index(drop=True)

print(df_resultados.to_string(index=False))

             Modelo Config  ROC-AUC (CV)  ROC-AUC (Test)  Accuracy  F1-Score
            XGBoost PCA-15      0.522661        0.515377  0.511328  0.610116
      Random Forest  PCA-5      0.512711        0.512455  0.517289  0.680353
Logistic Regression  Top-5      0.509424        0.510683  0.516494  0.681169


## 8. Resumen Final

Mejor modelo: XGBoost con PCA-15   

In [18]:
# Modelo ganador: XGBoost + PCA-15
print(f"\nMétricas de Optimización (Cross-Validation):")
print(f"  ROC-AUC (CV): {df_resultados.loc[0, 'ROC-AUC (CV)']:.4f}")
print(f"\nMétricas en Test Set:")
print(f"  ROC-AUC:   {df_resultados.loc[0, 'ROC-AUC (Test)']:.4f}")
print(f"  Accuracy:  {df_resultados.loc[0, 'Accuracy']:.4f}")
print(f"  F1-Score:  {df_resultados.loc[0, 'F1-Score']:.4f}")

# Matriz de Confusión del modelo XGBoost
print("Matriz de Confusión:")
cm = confusion_matrix(y_test, y_pred_xgb)
print(cm)
print(f"\nTrue Negatives (TN):  {cm[0,0]}")
print(f"False Positives (FP): {cm[0,1]}")
print(f"False Negatives (FN): {cm[1,0]}")
print(f"True Positives (TP):  {cm[1,1]}")

# Classification Report detallado
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb, target_names=['Vender (0)', 'Comprar (1)']))


Métricas de Optimización (Cross-Validation):
  ROC-AUC (CV): 0.5227

Métricas en Test Set:
  ROC-AUC:   0.5154
  Accuracy:  0.5113
  F1-Score:  0.6101
Matriz de Confusión:
[[ 649 1784]
 [ 675 1924]]

True Negatives (TN):  649
False Positives (FP): 1784
False Negatives (FN): 675
True Positives (TP):  1924

Classification Report:
              precision    recall  f1-score   support

  Vender (0)       0.49      0.27      0.35      2433
 Comprar (1)       0.52      0.74      0.61      2599

    accuracy                           0.51      5032
   macro avg       0.50      0.50      0.48      5032
weighted avg       0.51      0.51      0.48      5032

